# Lab 08: Exponential Smoothing

รายวิชา `229351: Statistical Learning for Data Science 1`

Lab นี้ใช้อนุกรม `CM_temp` และ `CarSales` ชุดเดียวกับที่ใช้ในบทเรียน Lec 08 ตัวเลขทุกตัวที่ได้จึงตรงกับตัวเลขบนสไลด์ ถ้าค่าที่ได้ไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด ไม่ใช่เรื่องของการปัดเศษ

Lab นี้เขียน recursion ของ exponential smoothing ด้วยมือก่อน แล้วจึงใช้ `statsmodels` เป็นตัวตรวจคำตอบ ไม่ใช่ในทางกลับกัน

## เป้าหมาย

1. เตรียมอนุกรมเวลาให้มี `DatetimeIndex` ที่มี frequency ชัดเจน ทั้งจากข้อมูลรายวันที่ต้อง `resample` และจากข้อมูลรายเดือนที่มีอยู่แล้ว
2. เขียน recursion ของ simple exponential smoothing (SES) ด้วยมือ และอธิบายบทบาทของ `l0` ได้
3. หา `alpha` ด้วยการค้นหาเชิงตัวเลขบน SSE และอธิบายได้ว่าเหตุใดจึงไม่มีสูตรปิดแบบ OLS
4. อธิบายได้ว่าเหตุใดการตรึง `l0` ไว้ที่ค่าแรกของอนุกรมจึงทำให้ `alpha` ที่ได้ผิด
5. fit Holt และ damped Holt อ่านค่าพารามิเตอร์ที่ติดขอบ และคำนวณเพดานของ damped trend ได้
6. fit Holt-Winters ทั้งแบบบวกและแบบคูณ และเลือก model จากทั้ง AICc และ test RMSE
7. ตรวจ residual ด้วย Ljung-Box และรายงาน prediction interval แทนค่าพยากรณ์จุดเดียว
8. เปรียบเทียบ ETS กับ benchmark ที่ไม่มีพารามิเตอร์ และสรุปผลอย่างซื่อสัตย์ต่อหลักฐาน

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.stats.diagnostic import acorr_ljungbox

pd.set_option("display.float_format", lambda value: f"{value:.6g}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

# ข้อมูลถูก pin ไว้ที่ commit หนึ่งโดยเฉพาะ เพื่อให้คำตอบเชิงตัวเลขไม่เปลี่ยนตามเวลา
PIN = "8d2d9c8ebfd86113c62a1f994f88e6e6732b0277"
BASE = f"https://raw.githubusercontent.com/skitimo/ds351-data/{PIN}"
print(BASE)

## 1. ข้อมูลและดัชนีเวลา (8 คะแนน)

Lab นี้ใช้สองอนุกรม

| ไฟล์ | เนื้อหา | ความถี่ดิบ |
|---|---|---|
| `CM_temp.csv` | อุณหภูมิสูงสุดในเชียงใหม่ 1998-01-01 ถึง 2019-07-31 | **รายวัน** 7882 แถว |
| `CarSales.csv` | ยอดขายรถยนต์ในควิเบก 1960-01 ถึง 1968-12 | รายเดือน 108 แถว |

`CM_temp` เป็นข้อมูลรายวัน แต่ทั้งสไลด์และ lab นี้ใช้ค่าเฉลี่ย **รายปี** และ **รายเดือน** ของมัน จึงต้อง `resample` ก่อน นี่เป็นขั้นตอนเตรียมข้อมูลจริง ไม่ใช่ขั้นตอนประดับ

**สามกับดักที่ต้องผ่านให้ได้ตั้งแต่เซลล์นี้**

| เขียนแบบนี้ | ผลลัพธ์ |
|---|---|
| `resample("M")` | `ValueError` ใน pandas 3 ให้ใช้ `"ME"` (สิ้นเดือน) หรือ `"MS"` (ต้นเดือน) |
| `read_csv(..., date_parser=...)` | `TypeError` ถูกถอดออกตั้งแต่ pandas 2.0 ให้ใช้ `parse_dates=` |
| ไม่ตั้ง frequency ให้ index | `ETSModel` จะตีความ `seasonal_periods` ผิด ให้เรียก `.asfreq(...)` ทุกครั้ง |

กับดักที่สามคือกับดักที่เงียบที่สุด เพราะมันไม่ error

In [ ]:
raw_temp = pd.read_csv(f"{BASE}/CM_temp.csv", parse_dates=["Date"])
raw_sales = pd.read_csv(f"{BASE}/CarSales.csv")
display(raw_temp.head())
display(raw_sales.head())
print(raw_temp.dtypes.to_dict(), raw_sales.dtypes.to_dict())

In [ ]:
# TODO: สร้าง temp_daily เป็น Series ของ MaxTemp ที่ index คือ Date
#       แล้ว resample เป็นค่าเฉลี่ยรายปี (temp_annual) และรายเดือน (temp_monthly)
#       temp_annual ให้ตัดปี 2019 ทิ้ง เพราะข้อมูลปีนั้นมีถึงแค่เดือนกรกฎาคม จึงไม่ใช่ค่าเฉลี่ยทั้งปี
#       อย่าลืม .asfreq(...) ปิดท้ายทั้งสองอนุกรม
#       คำแนะนำ: "YS" คือต้นปี "MS" คือต้นเดือน และ .loc[:"2018"] ตัดท้ายได้ด้วย string
temp_daily = ...
temp_annual = ...
temp_monthly = ...

print("daily  ", len(temp_daily), temp_daily.index.min().date(), temp_daily.index.max().date())
print("annual ", len(temp_annual), temp_annual.index.freqstr)
print("monthly", len(temp_monthly), temp_monthly.index.freqstr)

In [ ]:
# TODO: สร้าง sales เป็น Series ของ Sales ที่ index เป็น DatetimeIndex ต้นเดือน
#       column Month อยู่ในรูปแบบ "YYYY-MM"
#       คำแนะนำ: pd.PeriodIndex(..., freq="M").to_timestamp() แปลง "1960-01" เป็น 1960-01-01
sales_index = ...
sales = ...

print(sales.head(3))
print("n =", len(sales), "freq =", sales.index.freqstr)

In [ ]:
# ตรวจว่าทุกอนุกรมตรงกับที่สไลด์ใช้ ถ้า assert ใดไม่ผ่าน อย่าทำข้อถัดไปจนกว่าจะแก้ได้
assert len(temp_daily) == 7882, len(temp_daily)
assert temp_daily.isna().sum() == 0
assert len(temp_annual) == 21 and temp_annual.index.year.min() == 1998 and temp_annual.index.year.max() == 2018
assert len(temp_monthly) == 259
assert len(sales) == 108

print("annual  mean = %.3f  sd = %.3f" % (temp_annual.mean(), temp_annual.std()))
print("annual  min  = %.2f (%d)   max = %.2f (%d)" % (
    temp_annual.min(), temp_annual.idxmin().year, temp_annual.max(), temp_annual.idxmax().year))

# ตัวเลขสี่ตัวนี้อยู่บนสไลด์ Lec 08 หน้าแรกของหัวข้อ SES
assert abs(temp_annual.mean() - 32.526) < 0.001
assert abs(temp_annual.std() - 0.752) < 0.001
assert abs(temp_annual.min() - 31.49) < 0.01
assert abs(temp_annual.max() - 33.70) < 0.01

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(temp_annual.index, temp_annual.to_numpy(), marker="o", color="black", linewidth=1.2)
axes[0].set_title("CM_temp: annual mean of daily max temperature (1998-2018)")
axes[1].plot(temp_monthly.index, temp_monthly.to_numpy(), color="black", linewidth=0.9)
axes[1].set_title("CM_temp: monthly mean (1998-01 to 2019-07)")
plt.tight_layout(); plt.show()

### เขียนคำตอบ ส่วนที่ 1

1. `temp_annual` และ `temp_monthly` มีกี่จุดตามลำดับ และการรวมเป็นรายปีทำให้เสียข้อมูลอะไรไป แลกกับได้อะไรกลับมา
2. เหตุใดจึงต้องตัดปี 2019 ออกจาก `temp_annual` ถ้าไม่ตัดจะเกิดอะไรขึ้นกับค่าเฉลี่ยของปีนั้น และเหตุใดจึงกระทบการพยากรณ์เป็นพิเศษ
3. `.asfreq("YS")` เปลี่ยนค่าในอนุกรมหรือไม่ ถ้าไม่ แล้วมันเปลี่ยนอะไร และเหตุใด `ETSModel` จึงต้องการมัน

_เขียนคำตอบที่นี่_

## 2. Simple exponential smoothing

SES เขียนได้สองรูปแบบที่สมมูลกัน

**รูปแบบค่าเฉลี่ยถ่วงน้ำหนัก** ให้น้ำหนักลดลงแบบ geometric กับข้อมูลที่เก่าลง

$$\hat{y}_{T+1|T} = \alpha y_T + \alpha(1-\alpha) y_{T-1} + \alpha(1-\alpha)^2 y_{T-2} + \cdots$$

**รูปแบบเวียนเกิด** ซึ่งเป็นรูปแบบที่เขียน code ได้จริง

$$\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}, \qquad \hat{y}_{t|t-1} = \ell_{t-1}, \qquad \hat{y}_{T+h|T} = \ell_T$$

สังเกตดัชนีให้ดี **ค่าพยากรณ์ของคาบ $t$ คือ level ของคาบ $t-1$** นั่นคือใช้ข้อมูลถึงคาบ $t-1$ เท่านั้น ไม่ใช้ $y_t$ ซึ่งเป็นสิ่งที่เรากำลังจะทำนาย ดังนั้น $\hat{y}_1 = \ell_0$ พอดี เพราะยังไม่เห็นข้อมูลเลยแม้แต่จุดเดียว

$\ell_0$ จึงไม่ใช่ค่าที่ตั้งเล่น ๆ แต่เป็น **พารามิเตอร์ที่ต้องประมาณ** เท่าเทียมกับ $\alpha$ Exercise 2 จะแสดงว่าการลืมข้อนี้ทำให้คำตอบผิดไปเกินเท่าตัว

### Exercise 1: เขียน recursion ของ SES ด้วยมือ (12 คะแนน)

In [ ]:
# TODO: เขียนฟังก์ชัน ses_run(y, alpha, l0)
#       คืนค่าสองอนุกรมความยาวเท่ากับ y คือ
#         fitted[t] = yhat_t  ซึ่งคือ level "ก่อน" เห็น y[t]
#         levels[t] = l_t     ซึ่งคือ level "หลัง" เห็น y[t] แล้ว
#       ห้ามใช้ statsmodels ในเซลล์นี้
def ses_run(y, alpha, l0):
    y = np.asarray(y, dtype=float)
    fitted = np.empty(len(y))
    levels = np.empty(len(y))
    level = l0
    for t in range(len(y)):
        ...
    return fitted, levels

print(ses_run(np.array([1.0, 2.0, 3.0]), 0.5, 0.0))

In [ ]:
# ค่าสองตัวนี้คือค่าที่ Lec 08 fit ได้ Exercise 2 จะหามันเองตั้งแต่ต้น
ALPHA_HAT = 0.3143
L0_HAT = 32.3028

fitted, levels = ses_run(temp_annual, ALPHA_HAT, L0_HAT)
recursion = pd.DataFrame({
    "year": temp_annual.index.year,
    "y_t": temp_annual.to_numpy(),
    "yhat_t": fitted,
    "e_t": temp_annual.to_numpy() - fitted,
    "l_t": levels,
}, index=np.arange(1, len(temp_annual) + 1))
recursion.index.name = "t"
display(recursion.round(3).head(5))
display(recursion.round(3).tail(1))

sse_hand = float(((temp_annual.to_numpy() - fitted) ** 2).sum())
print("SSE =", round(sse_hand, 4))

In [ ]:
# ตรวจกับตารางบนสไลด์ Lec 08 ทุกค่าต้องตรงภายใน 0.01
assert abs(recursion.loc[1, "yhat_t"] - L0_HAT) < 1e-12, "yhat_1 ต้องเท่ากับ l0 พอดี"
for t, y_t, yhat_t, e_t, l_t in [
        (1, 33.579, 32.303,  1.276, 32.704),
        (2, 31.591, 32.704, -1.112, 32.354),
        (3, 31.493, 32.354, -0.861, 32.083),
        (4, 31.680, 32.083, -0.404, 31.957),
        (5, 31.808, 31.957, -0.149, 31.910),
        (21, 32.972, 33.154, -0.182, 33.097)]:
    row = recursion.loc[t]
    assert abs(row["y_t"] - y_t) < 0.01, (t, "y")
    assert abs(row["yhat_t"] - yhat_t) < 0.01, (t, "yhat")
    assert abs(row["e_t"] - e_t) < 0.01, (t, "e")
    assert abs(row["l_t"] - l_t) < 0.01, (t, "l")
assert abs(sse_hand - 9.3642) < 0.001
print("ตรงกับสไลด์ทุกค่า")

In [ ]:
# TODO: สร้าง forecast 5 ปีข้างหน้า (2019-2023) จาก level ตัวสุดท้าย
#       SES ไม่มี trend และไม่มี seasonal ค่าพยากรณ์จึงเท่ากันทุก h
h_index = pd.period_range("2019", periods=5, freq="Y").to_timestamp()
forecast_ses = ...

print(forecast_ses.round(3))
assert forecast_ses.nunique() == 1, "ค่าพยากรณ์ของ SES ต้องเท่ากันทุก h"
assert abs(forecast_ses.iloc[0] - 33.097) < 0.01

In [ ]:
fig, ax = plt.subplots()
ax.plot(temp_annual.index, temp_annual.to_numpy(), marker="o", color="black",
        linewidth=1.2, label="actual")
ax.plot(temp_annual.index, fitted, color="#0E7C7B", linewidth=1.4, label="one-step fitted")
ax.plot(forecast_ses.index, forecast_ses.to_numpy(), color="#A23E48",
        linewidth=1.6, marker="s", markersize=4, label="forecast")
ax.set_title(f"SES on CM_temp annual, alpha = {ALPHA_HAT}")
ax.legend(); plt.show()

### เขียนคำตอบ Exercise 1

1. เหตุใด $\hat{y}_1$ จึงเท่ากับ $\ell_0$ พอดี ถ้าเขียน code แล้วได้ $\hat{y}_1 \ne \ell_0$ แปลว่าเขียนลูปผิดตรงไหน
2. ดูกราฟแล้วอธิบายความสัมพันธ์ระหว่างเส้น fitted กับเส้นข้อมูลจริง โดยอ้างค่า $\alpha$ และแถว $t = 1$ ในตาราง
3. ค่าพยากรณ์ทั้ง 5 ปีเท่ากันหมด เหตุใดจึงเป็นเช่นนั้น ให้ตอบโดยอ้างสมการพยากรณ์ ไม่ใช่รูปร่างของกราฟ
4. $\ell_{21} = 33.097$ สูงกว่า $y_{21} = 32.972$ เหตุใด level จึงสูงกว่าค่าจริงล่าสุดได้

_เขียนคำตอบที่นี่_

## 3. หา `alpha` ด้วยการค้นหาเชิงตัวเลข

ใน linear regression เราหาค่าสัมประสิทธิ์ที่ทำให้ SSE ต่ำที่สุดด้วย **สูตรปิด** $\hat\beta = (X^\top X)^{-1}X^\top y$ ใน SES ไม่มีสูตรแบบนั้น เพราะ $\alpha$ อยู่ใน recursion แบบทบต้น ค่าพยากรณ์ที่คาบ $t$ ขึ้นกับ $(1-\alpha)^{t-1}$ จึงเป็นฟังก์ชันไม่เชิงเส้นของ $\alpha$ ทางเดียวที่เหลือคือ **ค้นหา**

$$\min_{\alpha,\,\ell_0} \; \mathrm{SSE}(\alpha, \ell_0) = \sum_{t=1}^{T} \bigl(y_t - \hat{y}_{t|t-1}\bigr)^2$$

สังเกตว่าเราต้องหาให้ครบ **สองตัว** ไม่ใช่ตัวเดียว และนี่คือจุดที่ tutorial ส่วนใหญ่บนอินเทอร์เน็ตทำผิด

**ข่าวดีคือ $\ell_0$ มีสูตรปิด เมื่อ $\alpha$ ถูกตรึงไว้แล้ว** เพราะ $\ell_t$ เป็นฟังก์ชัน affine ของ $\ell_0$ เขียน $\hat{y}_t = a_t + w_t \ell_0$ โดยที่ $w_t = (1-\alpha)^{t-1}$ และ $a_t$ คือค่าที่ได้จากการรัน recursion ด้วย $\ell_0 = 0$ ความคลาดเคลื่อนจึงเป็น

$$e_t = y_t - a_t - w_t\ell_0 = c_t - w_t \ell_0, \qquad c_t = y_t - a_t$$

SSE เป็นพหุนามกำลังสองใน $\ell_0$ อนุพันธ์เป็นศูนย์ให้

$$\ell_0^{*}(\alpha) = \frac{\sum_t w_t c_t}{\sum_t w_t^{2}} = \frac{w \cdot c}{w \cdot w}$$

เราจึงค้นหาเฉพาะ $\alpha$ บน grid หนึ่งมิติ แล้วให้ $\ell_0$ ตามด้วยสูตรนี้ทุกครั้ง **หนึ่งพารามิเตอร์มีสูตรปิด อีกหนึ่งไม่มี** ซึ่งเป็นภาพรวมของทั้งบท

### Exercise 2: SSE curve, `alpha` ที่ดีที่สุด และกับดักของ `l0` (18 คะแนน)

In [ ]:
# TODO: เขียน geometric_weights(alpha, n) คืน array ของน้ำหนัก alpha*(1-alpha)**j สำหรับ j = 0..n-1
#       และ half_life(alpha) คืน log(0.5)/log(1-alpha) ซึ่งคือจำนวนคาบที่น้ำหนักลดลงครึ่งหนึ่ง
def geometric_weights(alpha, n):
    j = np.arange(n)
    return ...

def half_life(alpha):
    return ...

weights = pd.DataFrame(
    [dict(alpha=a, **{f"j={j}": v for j, v in enumerate(geometric_weights(a, 4))},
          sum_j_le_3=geometric_weights(a, 4).sum(), half_life=half_life(a))
     for a in (0.2, 0.4, 0.8)]).set_index("alpha")
display(weights.round(3))

assert abs(weights.loc[0.2, "half_life"] - 3.11) < 0.01
assert abs(weights.loc[0.8, "sum_j_le_3"] - 0.998) < 0.001

In [ ]:
# TODO: เขียน best_l0(y, alpha) ตามสูตรปิดข้างบน
#       บรรทัดแรกให้มาแล้ว คือการรัน recursion ด้วย l0 = 0 เพื่อให้ได้ a_t
def best_l0(y, alpha):
    y = np.asarray(y, dtype=float)
    a_t, _ = ses_run(y, alpha, 0.0)
    c = ...
    w = ...
    return ...

def sse_at(y, alpha, l0):
    """SSE ของ SES ที่ค่า alpha และ l0 ที่กำหนด"""
    fit, _ = ses_run(y, alpha, l0)
    return float(((np.asarray(y, dtype=float) - fit) ** 2).sum())

print("l0*(0.314) =", round(best_l0(temp_annual, 0.314), 4))

In [ ]:
# ตาราง SSE ที่ค่า alpha เจ็ดค่า โดยหา l0 ใหม่ทุกครั้ง -- ตารางนี้อยู่บนสไลด์ Lec 08
grid_alpha = np.array([0.05, 0.10, 0.20, 0.314, 0.50, 0.80, 0.95])
grid_table = pd.DataFrame({
    "alpha": grid_alpha,
    "l0_star": [best_l0(temp_annual, a) for a in grid_alpha],
    "SSE": [sse_at(temp_annual, a, best_l0(temp_annual, a)) for a in grid_alpha],
}).set_index("alpha")
display(grid_table.round(3))

for a, sse_expected in [(0.05, 11.376), (0.10, 10.762), (0.20, 9.673), (0.314, 9.364),
                        (0.50, 9.725), (0.80, 11.012), (0.95, 11.938)]:
    assert abs(grid_table.loc[a, "SSE"] - sse_expected) < 0.001, a
print("ตรงกับตารางบนสไลด์ทั้งเจ็ดแถว")

In [ ]:
# TODO: ค้นหา alpha ที่ดีที่สุดบน grid ละเอียด โดยหา l0 ใหม่ทุกค่าของ alpha
alpha_fine = np.round(np.arange(0.01, 1.00, 0.001), 3)
sse_opt = ...
alpha_star = ...
l0_star = ...

print("alpha* = %.3f   SSE = %.4f   l0* = %.4f" % (alpha_star, sse_opt.min(), l0_star))
assert abs(alpha_star - 0.314) < 0.002
assert abs(sse_opt.min() - 9.3642) < 0.001

In [ ]:
# TODO: ทำซ้ำอีกครั้ง แต่คราวนี้ "ตรึง" l0 ไว้ที่ค่าแรกของอนุกรม ซึ่งเป็นวิธีที่ tutorial ส่วนใหญ่ใช้
#       ใช้ grid เดียวกัน เปลี่ยนแค่วิธีได้มาซึ่ง l0
sse_fixed = ...
alpha_fixed = ...

print("l0 = y[0] = %.3f" % temp_annual.iloc[0])
print("alpha (l0 ตรึง) = %.3f   SSE = %.4f" % (alpha_fixed, sse_fixed.min()))
print("alpha (l0 หาใหม่) = %.3f   SSE = %.4f" % (alpha_star, sse_opt.min()))
print("ผิดไป %.2f เท่า" % (alpha_fixed / alpha_star))

assert abs(alpha_fixed - 0.618) < 0.005
assert alpha_fixed > 1.9 * alpha_star

In [ ]:
fig, ax = plt.subplots()
ax.plot(alpha_fine, sse_opt, color="#0E7C7B", linewidth=1.8, label="l0 หาใหม่ทุก alpha (ถูก)")
ax.plot(alpha_fine, sse_fixed, color="#A23E48", linewidth=1.8, linestyle="--", label="l0 = y[0] (ผิด)")
ax.plot([alpha_star], [sse_opt.min()], marker="o", color="#0E7C7B", markersize=8)
ax.plot([alpha_fixed], [sse_fixed.min()], marker="o", color="#A23E48", markersize=8)
ax.set_xlabel("alpha"); ax.set_ylabel("SSE")
ax.set_title("Both curves are smooth. Only one has its minimum in the right place.")
ax.legend(); plt.show()

In [ ]:
# ตรวจคำตอบด้วย ETSModel ซึ่ง optimise alpha และ l0 พร้อมกันด้วย likelihood
ses_fit = ETSModel(temp_annual, error="add", trend=None, seasonal=None).fit(disp=False)
ses_l0 = ses_fit.params[ses_fit.param_names.index("initial_level")]

print("ETSModel : alpha = %.4f   l0 = %.4f   SSE = %.4f   AICc = %.2f" % (
    ses_fit.smoothing_level, ses_l0, ses_fit.sse, ses_fit.aicc))
print("grid     : alpha = %.4f   l0 = %.4f   SSE = %.4f" % (alpha_star, l0_star, sse_opt.min()))

assert abs(ses_fit.smoothing_level - 0.3143) < 0.001
assert abs(ses_fit.sse - 9.3642) < 0.001
assert abs(ses_fit.sse - sse_opt.min()) < 0.001, "grid ที่เขียนเองต้องได้ SSE เท่ากับ library"
assert abs(ses_fit.aicc - 50.05) < 0.01

### เขียนคำตอบ Exercise 2

1. จากตารางน้ำหนัก เปรียบเทียบ half-life ของ $\alpha = 0.2$ กับ $\alpha = 0.8$ และอธิบายว่าตัวเลขนี้บอกอะไรเกี่ยวกับ "ความจำ" ของ model
2. $\ell_0$ มีสูตรปิดเมื่อตรึง $\alpha$ ไว้แล้ว แต่ $\alpha$ ไม่มีสูตรปิด เหตุใดจึงเป็นเช่นนั้น ให้อ้างรูปแบบที่ $\alpha$ ปรากฏในสมการ
3. เมื่อตรึง $\ell_0 = y_1$ ค่า $\alpha$ ที่ได้คือเท่าไร และ**เพราะเหตุใด**การตรึงจึงดัน $\alpha$ ขึ้นแทนที่จะลง
4. ความผิดพลาดแบบนี้อันตรายกว่าหรือน้อยกว่า code ที่ error แล้วหยุดทำงาน เพราะอะไร
5. ค่าที่ `ETSModel` หาได้ตรงกับ grid ที่เขียนเองหรือไม่ ถ้าต่างกัน ต่างกันตรงไหนและเพราะอะไร

_เขียนคำตอบที่นี่_

## 4. Holt และ damped Holt

SES มีสมการเดียว Holt เพิ่มสมการที่สองเข้ามาเพื่อให้ระดับเคลื่อนที่ได้

$$\ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1} + b_{t-1}), \qquad
b_t = \beta^{*}(\ell_t - \ell_{t-1}) + (1-\beta^{*})b_{t-1}$$

$$\hat{y}_{T+h|T} = \ell_T + h\,b_T$$

ค่าพยากรณ์กลายเป็นเส้นตรงที่ยิงไปไม่สิ้นสุด ซึ่งแทบไม่เคยสมจริง **damped Holt** จึงใส่ $\phi \in (0,1)$ เข้าไปเบรก

$$\hat{y}_{T+h|T} = \ell_T + (\phi + \phi^2 + \cdots + \phi^h)\,b_T
\;\xrightarrow[h \to \infty]{}\; \ell_T + \frac{\phi}{1-\phi}b_T$$

ค่าพยากรณ์ระยะยาวจึงลู่เข้า **เพดาน** ที่คำนวณได้ล่วงหน้า แทนที่จะวิ่งไปหาอนันต์

### Exercise 3: trend จ่ายค่าพารามิเตอร์ของตัวเองหรือไม่ (14 คะแนน)

Holt มีพารามิเตอร์มากกว่า SES อยู่สองตัว ($\beta^{*}$ และ $b_0$) SSE จึงต้องลดลงเสมอ คำถามที่ต้องตอบคือ **ลดลงมากพอจะคุ้มกับพารามิเตอร์ที่เพิ่มหรือไม่** ซึ่งเป็นสิ่งที่ AICc ตอบ ไม่ใช่ SSE

In [ ]:
# TODO: fit Holt (ไม่ damped) บน temp_annual ด้วย ETSModel แล้วดึงพารามิเตอร์ออกมา
#       trend="add" คือการเปิด slot ของ trend  ห้ามเขียน trend="none" ให้ใช้ None
holt_fit = ...
holt_params = dict(zip(holt_fit.param_names, holt_fit.params))

print("alpha = %.4f   beta = %.4f" % (holt_fit.smoothing_level, holt_fit.smoothing_trend))
print("l0    = %.4f   b0   = %.4f" % (holt_params["initial_level"], holt_params["initial_trend"]))
print("SSE   = %.4f   AICc = %.2f" % (holt_fit.sse, holt_fit.aicc))

assert abs(holt_fit.sse - 7.276) < 0.01
assert abs(holt_fit.aicc - 51.34) < 0.05

In [ ]:
# SSE ลดลง แต่ AICc แย่ลง -- ตารางนี้คือคำตอบของ Exercise 3
compare = pd.DataFrame({
    "SSE": [ses_fit.sse, holt_fit.sse],
    "AICc": [ses_fit.aicc, holt_fit.aicc],
    "k (พารามิเตอร์)": [len(ses_fit.params), len(holt_fit.params)],
}, index=["SES", "Holt"])
display(compare.round(3))

assert holt_fit.sse < ses_fit.sse, "Holt มีพารามิเตอร์มากกว่า SSE จึงต้องลดลงเสมอ"
assert holt_fit.aicc > ses_fit.aicc, "แต่ AICc แย่ลง -- trend ไม่คุ้มค่าพารามิเตอร์บนอนุกรมนี้"
print("SSE ลดลง %.3f แต่ AICc แย่ลง %.2f" % (
    ses_fit.sse - holt_fit.sse, holt_fit.aicc - ses_fit.aicc))

In [ ]:
# beta ถูก fit ไว้ที่ 0 พอดี ซึ่งหมายความว่าความชันไม่เคยถูกปรับเลยตลอดอนุกรม
# Holt จึงยุบลงเป็น "เส้นตรงตายตัว" เทียบกับ OLS ที่ถดถอย y บนดัชนีเวลา
ols_slope, ols_intercept = np.polyfit(np.arange(len(temp_annual)), temp_annual.to_numpy(), 1)

print("Holt b0    = %.4f" % holt_params["initial_trend"])
print("OLS slope  = %.4f" % ols_slope)
assert abs(holt_params["initial_trend"] - ols_slope) < 0.002, "beta = 0 ทำให้ Holt กลายเป็น OLS บนเวลา"
print("เท่ากันทุกตำแหน่งทศนิยมที่แสดง")

In [ ]:
# TODO: เขียน damped_path(l_T, b_T, phi, h_max) คืน array ของค่าพยากรณ์ h = 1..h_max
#       ตามสูตร l_T + (phi + phi^2 + ... + phi^h) * b_T
#       และ damped_ceiling(l_T, b_T, phi) คืน l_T + b_T*phi/(1-phi)
def damped_path(l_T, b_T, phi, h_max):
    h = np.arange(1, h_max + 1)
    damping_sum = ...
    return ...

def damped_ceiling(l_T, b_T, phi):
    return ...

# state ของ Holt บน AirPassengers รายปี ณ ปี 1960 -- ค่าสองตัวนี้อยู่บนสไลด์ Lec 08
L_T, B_T = 5688.71, 381.44

damped = pd.DataFrame(
    {f"phi={phi}": damped_path(L_T, B_T, phi, 15)[[0, 4, 9, 14]] for phi in (1.0, 0.9, 0.8)},
    index=[1, 5, 10, 15])
damped.index.name = "h"
display(damped.round(1))
print("ceiling phi=0.9 : %.1f" % damped_ceiling(L_T, B_T, 0.9))
print("ceiling phi=0.8 : %.1f" % damped_ceiling(L_T, B_T, 0.8))

In [ ]:
# ตรวจกับตารางบนสไลด์
for phi, h, expected in [(1.0, 1, 6070.2), (1.0, 15, 11410.4),
                         (0.9, 5, 7094.6), (0.9, 15, 8414.9),
                         (0.8, 10, 7050.7), (0.8, 15, 7160.8)]:
    got = damped.loc[h, f"phi={phi}"]
    assert abs(got - expected) < 0.2, (phi, h, got, expected)
assert abs(damped_ceiling(L_T, B_T, 0.9) - 9121.7) < 0.2
assert abs(damped_ceiling(L_T, B_T, 0.8) - 7214.5) < 0.2

# phi = 1 คือ Holt ธรรมดา ซึ่งไม่มีเพดาน
assert damped.loc[15, "phi=1.0"] > damped.loc[15, "phi=0.9"] > damped.loc[15, "phi=0.8"]
print("ตรงกับสไลด์ทุกค่า")

In [ ]:
fig, ax = plt.subplots()
h_axis = np.arange(1, 41)
for phi, colour in [(1.0, "#243B53"), (0.9, "#0E7C7B"), (0.8, "#B68D40")]:
    ax.plot(h_axis, damped_path(L_T, B_T, phi, 40), color=colour, linewidth=1.6, label=f"phi = {phi}")
    if phi < 1.0:
        ax.axhline(damped_ceiling(L_T, B_T, phi), color=colour, linestyle=":", linewidth=1.2)
ax.set_xlabel("h (periods ahead)"); ax.set_ylabel("forecast")
ax.set_title("Damped trend converges to a ceiling; phi = 1 does not")
ax.legend(); plt.show()

### เขียนคำตอบ Exercise 3

1. SSE ของ Holt ต่ำกว่า SES แต่ AICc แย่กว่า ตัวเลขสองตัวนี้ขัดแย้งกันหรือไม่ อธิบายว่าแต่ละตัววัดอะไร และสรุปว่าควรเลือก model ใด
2. $\hat\beta^{*} = 0$ แปลว่าอะไรในเชิงสมการ และเหตุใดพารามิเตอร์ที่ถูก fit ไปติดขอบจึงเป็น "ข้อมูลกำลังบอกอะไรบางอย่าง" ไม่ใช่ error
3. $b_0$ ของ Holt เท่ากับความชันของ OLS บนดัชนีเวลาพอดี เหตุใดจึงเป็นเช่นนั้น และเหตุใดผลนี้จึงน่าสนใจเมื่อนึกถึงสิ่งที่ Lec 07 พูดถึงการ fit เส้นตรงบนอนุกรมเวลา
4. เพดานของ $\phi = 0.9$ กับ $\phi = 0.8$ ต่างกันเท่าไร และเหตุใดการเปลี่ยน $\phi$ เพียง $0.1$ จึงเปลี่ยนเพดานได้มากขนาดนั้น เหตุใด $\phi = 1$ จึงไม่มีเพดาน
5. เปรียบเทียบค่าพยากรณ์ของทั้งสาม $\phi$ ที่ $h = 1$ กับที่ $h = 15$ การเลือก $\phi$ สำคัญมากที่สุดเมื่อใด และเรื่องนี้มีความหมายอย่างไรในทางปฏิบัติ

_เขียนคำตอบที่นี่_

## 5. Holt-Winters และตาราง ETS

สมการที่สามคือ seasonal ซึ่งมีสองแบบ **แบบบวก** เหมาะเมื่อช่วงแกว่งตามฤดูกาลกว้างเท่าเดิมไม่ว่าอนุกรมจะอยู่ระดับใด และ **แบบคูณ** เหมาะเมื่อช่วงแกว่งโตตามระดับ

$$\hat{y}_{t+h|t} = \ell_t + h b_t + s_{t+h-m(k+1)}, \qquad k = \left\lfloor \frac{h-1}{m} \right\rfloor$$

ดัชนีของ $s$ ต้องนับให้ถูก เรามีสถานะ seasonal เพียง **หนึ่งรอบเต็มล่าสุด** เท่านั้น เมื่อ $h > m$ จึงต้องวนกลับไปใช้ดัชนีที่เคยใช้ไปแล้ว ซึ่งคือหน้าที่ของพจน์ $k$

ทั้งหมดนี้รวมอยู่ในตาราง **ETS** ที่ระบุด้วยสามตัวอักษร $(E, T, S)$ คือ Error, Trend, Seasonal แต่ละช่องเป็น `N` (ไม่มี), `A` (บวก), `M` (คูณ) และ `Ad`/`Md` (damped) รวม 30 แบบ

### Exercise 4: เลือก model บน `CarSales` (18 คะแนน)

`CarSales` คือยอดขายรถยนต์รายเดือนในควิเบก แบ่ง train เป็นปี 1960-1967 และ test เป็น 12 เดือนของปี 1968 ซึ่งเป็นช่วงเดียวกับที่สไลด์ Lec 08 ใช้

In [ ]:
sales_train, sales_test = sales[:-12], sales[-12:]
print("train %d เดือน  %s .. %s" % (len(sales_train), sales_train.index.min().date(), sales_train.index.max().date()))
print("test  %d เดือน  %s .. %s" % (len(sales_test), sales_test.index.min().date(), sales_test.index.max().date()))

assert len(sales_train) == 96 and len(sales_test) == 12

fig, ax = plt.subplots()
ax.plot(sales.index, sales.to_numpy(), color="black", linewidth=1.1)
ax.axvline(sales_train.index[-1], color="grey", linestyle="--", linewidth=1)
ax.set_title("CarSales, monthly; dashed line = end of train")
plt.show()

In [ ]:
# หกแบบเดียวกับที่สไลด์ Lec 08 ใช้ ทั้งบน AirPassengers และบน CarSales
ETS_SPECS = [
    ("(A,N,A)",  "add", None,  "add", False),
    ("(A,A,A)",  "add", "add", "add", False),
    ("(A,Ad,A)", "add", "add", "add", True),
    ("(A,A,M)",  "add", "add", "mul", False),
    ("(M,A,M)",  "mul", "add", "mul", False),
    ("(M,Ad,M)", "mul", "add", "mul", True),
]

# TODO: เขียน fit_ets_grid(series, h_test) ให้ fit ทั้งหกแบบบน train แล้วคืน DataFrame
#       ที่มี column AICc และ test RMSE  โดย train คือทุกอย่างยกเว้น h_test จุดสุดท้าย
#       เติมเฉพาะสองบรรทัดที่เป็น ... เท่านั้น
def fit_ets_grid(series, h_test):
    train, test = series[:-h_test], series[-h_test:]
    rows = []
    for name, error, trend, seasonal, damped in ETS_SPECS:
        fit = ...
        forecast = ...
        rows.append({"ETS": name, "AICc": fit.aicc,
                     "test RMSE": float(np.sqrt(((test.to_numpy() - forecast.to_numpy()) ** 2).mean()))})
    return pd.DataFrame(rows).set_index("ETS")

sales_grid = fit_ets_grid(sales, 12)
display(sales_grid.round(2))

In [ ]:
# ตรวจกับตารางบนสไลด์ Lec 08 ทั้งสิบสองค่า
expected_sales = {
    "(A,N,A)":  (1824.83, 2428.81),
    "(A,A,A)":  (1830.20, 2168.80),
    "(A,Ad,A)": (1833.86, 2387.14),
    "(A,A,M)":  (1700.59, 2025.32),
    "(M,A,M)":  (1710.69, 2035.24),
    "(M,Ad,M)": (1714.78, 2201.65),
}
for name, (aicc, rmse) in expected_sales.items():
    assert abs(sales_grid.loc[name, "AICc"] - aicc) < 0.05, (name, "AICc")
    assert abs(sales_grid.loc[name, "test RMSE"] - rmse) < 0.05, (name, "RMSE")

best_by_aicc = sales_grid["AICc"].idxmin()
best_by_rmse = sales_grid["test RMSE"].idxmin()
print("AICc เลือก:", best_by_aicc)
print("RMSE เลือก:", best_by_rmse)
assert best_by_aicc == best_by_rmse == "(A,A,M)"
print("บนอนุกรมนี้ทั้งสองเกณฑ์เห็นตรงกัน")

In [ ]:
# TODO: เขียน seasonal_source(last_obs, h, m) คืนเดือนที่ค่า s ถูกดึงมาใช้ ตามสูตร t + h - m*(k+1)
#       โดย k = (h-1) // m  และ last_obs คือคาบสุดท้ายของ train
#       คำแนะนำ: pd.DateOffset(months=n) บวกเดือนเข้ากับ Timestamp ได้
def seasonal_source(last_obs, h, m=12):
    k = ...
    return ...

t_last = pd.Timestamp("2019-01-01")
index_map = pd.DataFrame([
    {"h": h,
     "เดือนที่ทำนาย": (t_last + pd.DateOffset(months=h)).strftime("%Y-%m"),
     "s ที่ใช้": seasonal_source(t_last, h).strftime("%Y-%m")}
    for h in [1, 2, 11, 12, 13]]).set_index("h")
display(index_map)

assert index_map.loc[1, "s ที่ใช้"] == "2018-02"
assert index_map.loc[12, "s ที่ใช้"] == "2019-01"
assert index_map.loc[13, "s ที่ใช้"] == "2018-02", "h = 13 ต้องวนกลับไปใช้ดัชนีเดียวกับ h = 1"

In [ ]:
best_fit = ETSModel(sales_train, error="add", trend="add", seasonal="mul",
                    seasonal_periods=12).fit(disp=False)
best_forecast = best_fit.forecast(12)

fig, ax = plt.subplots()
ax.plot(sales.loc["1965":].index, sales.loc["1965":].to_numpy(), color="black",
        linewidth=1.1, label="actual")
ax.plot(best_forecast.index, best_forecast.to_numpy(), color="#A23E48",
        linewidth=1.6, marker="o", markersize=3, label="(A,A,M) forecast")
ax.axvline(sales_train.index[-1], color="grey", linestyle="--", linewidth=1)
ax.set_title("CarSales: the model both criteria chose")
ax.legend(); plt.show()

print("alpha = %.4f  beta = %.4f  gamma = %.4f" % (
    best_fit.smoothing_level, best_fit.smoothing_trend, best_fit.smoothing_seasonal))

### เขียนคำตอบ Exercise 4

1. AICc เลือก model ใด และ test RMSE เลือก model ใด ทั้งสองตรงกันหรือไม่ เทียบกับผลบน AirPassengers และ `elecequip` ที่อยู่บนสไลด์ แล้วสรุปว่าการที่สองเกณฑ์ตรงกันเป็นกฎหรือเป็นเรื่องเฉพาะชุดข้อมูล
2. เปรียบเทียบ AICc ของกลุ่มที่ใช้ seasonal แบบบวกกับกลุ่มที่ใช้แบบคูณ ห่างกันเท่าไร และตัวเลขนี้บอกอะไรเกี่ยวกับรูปร่างของ `CarSales`
3. ความต่างของ AICc ระหว่างแบบบวกกับแบบคูณ เทียบกับความต่างระหว่าง damped กับไม่ damped อย่างไหนสำคัญกว่า และมากกว่ากันประมาณกี่เท่า
4. จากตาราง `index_map` เมื่อ $h = 13$ model ใช้ดัชนีฤดูกาลของเดือนใด เพราะเหตุใด และข้อจำกัดนี้ส่งผลอย่างไรต่อการพยากรณ์ปีที่สอง
5. ดูค่า `alpha`, `beta`, `gamma` ของ model ที่ชนะ มีตัวใดติดขอบหรือไม่ ถ้ามี แปลว่าอะไร

_เขียนคำตอบที่นี่_

## 6. Residual และความไม่แน่นอน

ค่าพยากรณ์จุดเดียวไม่พอสำหรับการตัดสินใจ เราต้องรู้ด้วยว่ามันแม่นแค่ไหน และต้องตรวจก่อนว่า model ดูดซับโครงสร้างไปหมดแล้วจริงหรือไม่

**Residual ที่ดีต้องไม่มี autocorrelation เหลืออยู่** ถ้ายังมี แปลว่ายังมีรูปแบบที่ model มองไม่เห็น และเรายังปรับปรุงได้อีก **Ljung-Box test** ตรวจข้อนี้โดยมีสมมติฐานหลักว่า *ไม่มี* autocorrelation ดังนั้น **p-value สูงคือข่าวดี** ซึ่งกลับด้านกับ test ส่วนใหญ่ที่เคยเจอ

Prediction interval ของ SES ที่ระดับ 95% คือ $\hat{y}_{T+h|T} \pm 1.96\,\sigma_h$ โดยที่

$$\sigma_h = \sigma\sqrt{1 + (h-1)\alpha^2}$$

ช่วงจึงกว้างขึ้นเมื่อ $h$ โต แม้ค่าพยากรณ์จุดจะเท่าเดิมทุก $h$ ก็ตาม

**`holtwinters` ทำข้อนี้ไม่ได้เลย** `SimpleExpSmoothing`, `Holt` และ `ExponentialSmoothing` ไม่มี `get_prediction` นี่คือเหตุผลหลักที่ทั้ง lab และทั้งสไลด์ใช้ `ETSModel` อย่างเดียว

### Exercise 5: residual diagnostics และ prediction interval (12 คะแนน)

In [ ]:
# TODO: ดึง residual ของ ses_fit ออกมา คำนวณค่าเฉลี่ยกับส่วนเบี่ยงเบนมาตรฐาน
#       แล้วรัน Ljung-Box ที่ lag 5 และ 10
#       คำแนะนำ: acorr_ljungbox(..., lags=[5, 10], return_df=True)
ses_resid = ...
ljung = ...

print("mean = %+.4f   sd = %.4f" % (ses_resid.mean(), ses_resid.std()))
display(ljung.round(4))

assert abs(ses_resid.mean() - 0.1203) < 0.001
assert abs(ses_resid.std() - 0.6731) < 0.001
assert abs(ljung.loc[5, "lb_stat"] - 0.362) < 0.01
assert abs(ljung.loc[10, "lb_stat"] - 2.007) < 0.01
assert (ljung["lb_pvalue"] > 0.05).all(), "p สูงคือข่าวดี: ไม่มี autocorrelation เหลือ"

In [ ]:
# TODO: สร้าง prediction interval 95% สำหรับ 5 ปีข้างหน้า แล้วเทียบความกว้างกับสูตรทฤษฎี
#       sigma ประมาณด้วย sqrt(SSE / n)
#       คำแนะนำ: ses_fit.get_prediction(start=..., end=...).summary_frame(alpha=0.05)
#                จะให้ column mean, pi_lower, pi_upper
band = ...
sigma_hat = ...
band["width"] = band["pi_upper"] - band["pi_lower"]
band["theory_sd"] = [sigma_hat * np.sqrt(1 + (h - 1) * ses_fit.smoothing_level ** 2)
                     for h in range(1, 6)]

print("sigma = %.4f" % sigma_hat)
display(band.round(3))

assert abs(sigma_hat - 0.6678) < 0.001
assert abs(band["pi_lower"].iloc[0] - 31.788) < 0.01
assert abs(band["pi_upper"].iloc[0] - 34.406) < 0.01
assert abs(band["width"].iloc[0] - 2.618) < 0.01
assert abs(band["theory_sd"].iloc[4] - 0.7887) < 0.001
assert band["width"].is_monotonic_increasing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].axhline(0, color="grey", linewidth=1)
axes[0].plot(temp_annual.index, ses_resid.to_numpy(), marker="o", color="#243B53", linewidth=1)
axes[0].set_title("SES innovation residuals")

axes[1].plot(temp_annual.index, temp_annual.to_numpy(), color="black", linewidth=1.2, label="actual")
axes[1].plot(band.index, band["mean"], color="#A23E48", linewidth=1.6, label="forecast")
axes[1].fill_between(band.index, band["pi_lower"], band["pi_upper"],
                     color="#A23E48", alpha=0.18, label="95% PI")
axes[1].set_title("Point forecast is flat; the interval is not")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### เขียนคำตอบ Exercise 5

1. p-value ของ Ljung-Box ทั้งสอง lag เป็นเท่าไร และค่าสูงแบบนี้เป็นข่าวดีหรือข่าวร้าย เพราะเหตุใดจึงกลับด้านกับ test ที่เคยเรียนมา
2. ค่าเฉลี่ยของ residual ไม่เท่ากับศูนย์พอดี แปลว่าอะไร และเมื่อเทียบกับส่วนเบี่ยงเบนมาตรฐานแล้วควรกังวลหรือไม่
3. ค่าพยากรณ์จุดเท่ากันทุก $h$ แต่ความกว้างของช่วงไม่เท่า อธิบายว่าเหตุใดสองอย่างนี้จึงไม่ขัดแย้งกัน
4. จากสูตร $\sigma_h = \sigma\sqrt{1+(h-1)\alpha^2}$ ถ้า $\alpha$ ใกล้ 1 แทนที่จะเป็น $0.314$ ช่วงจะกว้างขึ้นเร็วกว่าหรือช้ากว่า และเหตุใดจึงสมเหตุสมผล
5. ถ้าใช้ `SimpleExpSmoothing` จาก `statsmodels.tsa.holtwinters` แทน จะสร้างตารางนี้ได้หรือไม่ เพราะอะไร

_เขียนคำตอบที่นี่_

## 7. โจทย์เปิด: อุณหภูมิเชียงใหม่รายเดือน

จนถึงตอนนี้ทุกข้อมีคำตอบที่ตรวจได้กับสไลด์ ข้อสุดท้ายไม่มี **ต้องตัดสินใจเองและปกป้องการตัดสินใจนั้นด้วยตัวเลข**

`temp_monthly` มีลักษณะตรงข้ามกับ `elecequip` ของ Lec 07 คือ **seasonality แรงมากแต่ trend เกือบแบน** อุณหภูมิเชียงใหม่วนรอบเดิมทุกปีอย่างสม่ำเสมอ และค่าเฉลี่ยรายปีแทบไม่ขยับในรอบ 20 ปี

แบ่ง test เป็น **24 เดือนสุดท้าย** ซึ่งยาวกว่าหนึ่งรอบฤดูกาล จึงเป็นการทดสอบข้อจำกัดที่เจอใน Exercise 4 ข้อ 4 โดยตรง

### Exercise 6: เลือก model และรายงานผลอย่างซื่อสัตย์ (18 คะแนน)

In [ ]:
monthly_train, monthly_test = temp_monthly[:-24], temp_monthly[-24:]
print("train %d เดือน  %s .. %s" % (
    len(monthly_train), monthly_train.index.min().date(), monthly_train.index.max().date()))
print("test  %d เดือน  %s .. %s" % (
    len(monthly_test), monthly_test.index.min().date(), monthly_test.index.max().date()))

assert len(monthly_train) == 235 and len(monthly_test) == 24
assert str(monthly_test.index.min().date()) == "2017-08-01"

In [ ]:
# TODO: ใช้ fit_ets_grid ที่เขียนไว้แล้วกับ temp_monthly โดยให้ test ยาว 24 เดือน
monthly_grid = ...
display(monthly_grid.round(3).sort_values("test RMSE"))

assert abs(monthly_grid.loc["(M,Ad,M)", "test RMSE"] - 1.1060) < 0.005
assert abs(monthly_grid.loc["(A,N,A)", "AICc"] - 720.13) < 0.1

In [ ]:
# TODO: เพิ่ม benchmark สองวิธีที่ไม่ต้อง fit อะไรเลย
#       seasonal naive = ทำซ้ำ 12 เดือนสุดท้ายของ train สองรอบ
#       mean = ค่าเฉลี่ยของ train ทั้งชุด
f_snaive = ...
f_mean = ...

def rmse(actual, predicted):
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(predicted)) ** 2)))

scoreboard = monthly_grid.copy()
scoreboard.loc["Seasonal naive", "test RMSE"] = rmse(monthly_test, f_snaive)
scoreboard.loc["Mean", "test RMSE"] = rmse(monthly_test, f_mean)
display(scoreboard.sort_values("test RMSE").round(3))

assert abs(scoreboard.loc["Seasonal naive", "test RMSE"] - 1.2227) < 0.005
assert abs(scoreboard.loc["Mean", "test RMSE"] - 2.6542) < 0.005

In [ ]:
# สองตัวเลขที่ Exercise 6 ข้อ 3 ต้องใช้
winner = scoreboard["test RMSE"].idxmin()
margin_over_snaive = scoreboard.loc["Seasonal naive", "test RMSE"] - scoreboard["test RMSE"].min()
spread_within_ets = monthly_grid["test RMSE"].max() - monthly_grid["test RMSE"].min()

print("ผู้ชนะ: %s ที่ RMSE = %.4f" % (winner, scoreboard["test RMSE"].min()))
print("ชนะ seasonal naive อยู่ %.4f องศา" % margin_over_snaive)
print("ช่วงห่างระหว่าง ETS ที่ดีที่สุดกับแย่ที่สุด = %.4f องศา" % spread_within_ets)
print("AICc เลือก: %s" % monthly_grid["AICc"].idxmin())

assert winner == "(M,Ad,M)"
assert monthly_grid["AICc"].idxmin() == winner, "บนชุดนี้สองเกณฑ์เห็นตรงกัน"
assert margin_over_snaive < 0.15

In [ ]:
best_monthly = ETSModel(monthly_train, error="mul", trend="add", damped_trend=True,
                        seasonal="mul", seasonal_periods=12).fit(disp=False)
fc_monthly = best_monthly.forecast(24)

fig, ax = plt.subplots()
recent = temp_monthly.loc["2014":]
ax.plot(recent.index, recent.to_numpy(), color="black", linewidth=1.1, label="actual")
ax.plot(fc_monthly.index, fc_monthly.to_numpy(), color="#A23E48", linewidth=1.5, label="(M,Ad,M)")
ax.plot(monthly_test.index, f_snaive, color="#B68D40", linewidth=1.3,
        linestyle="--", label="seasonal naive")
ax.axvline(monthly_train.index[-1], color="grey", linestyle="--", linewidth=1)
ax.set_title("24-month test: six fitted parameters vs copying last year")
ax.legend(fontsize=8); plt.show()

### เขียนคำตอบ Exercise 6

1. AICc เลือก model ใด และ test RMSE เลือก model ใด ตรงกันหรือไม่ เทียบกับผลใน Exercise 4 และผลบนสไลด์
2. ETS ทั้งหกแบบได้ RMSE ห่างกันเท่าไร และ `Seasonal naive` แพ้ผู้ชนะอยู่เท่าไร เทียบสองตัวเลขนี้กัน
3. `(M,Ad,M)` ใช้พารามิเตอร์ราว 17 ตัว ขณะที่ `Seasonal naive` ใช้ศูนย์ตัว จากตัวเลขในข้อ 2 การ fit นั้นคุ้มหรือไม่ ให้ตอบโดยเทียบกับสเกลของอนุกรม ไม่ใช่เทียบเฉพาะตัวเลข RMSE
4. เหตุใด `Seasonal naive` จึงทำได้ดีมากบนอนุกรมนี้ ให้ตอบโดยเทียบขนาดของ trend กับขนาดของ seasonality และเทียบกับ `elecequip` ใน Lab 07
5. เขียนหนึ่งย่อหน้าที่รายงานผลนี้ให้คนที่ไม่ได้ทำ lab อ่าน โดยต้องมีทั้งตัวเลข ทั้ง horizon ทั้งขนาดของชัยชนะ และทั้งข้อจำกัด

_เขียนคำตอบที่นี่_

## Checklist ก่อนส่ง

- [ ] โหลดข้อมูลจาก URL ที่ pin ไว้ และได้ `CM_temp` 7882 แถว กับ `CarSales` 108 แถว
- [ ] `temp_annual` มี 21 จุด (1998-2018) ตัดปี 2019 ออกแล้ว
- [ ] ทุกอนุกรมเรียก `.asfreq()` และ `index.freqstr` ไม่เป็น `None`
- [ ] `ses_run` ให้ `yhat_1 = l0` พอดี และ SSE = 9.3642
- [ ] grid ที่หา `l0` ใหม่ทุกครั้งได้ `alpha = 0.314` ส่วนที่ตรึง `l0 = y[0]` ได้ `0.618`
- [ ] `ETSModel` ให้ `alpha = 0.3143` และ SSE ตรงกับ grid ที่เขียนเอง
- [ ] Holt ได้ SSE ต่ำกว่า SES แต่ AICc แย่กว่า และ `beta = 0`
- [ ] เพดานของ damped ที่ `phi = 0.9` คือ 9121.7
- [ ] ตาราง ETS บน `CarSales` ตรงกับสไลด์ทั้ง 12 ค่า และทั้งสองเกณฑ์เลือก `(A,A,M)`
- [ ] `h = 13` ใช้ดัชนีฤดูกาลของ 2018-02 เหมือน `h = 1`
- [ ] Ljung-Box ได้ p สูงทั้งสอง lag และตอบได้ว่าเหตุใดจึงเป็นข่าวดี
- [ ] prediction interval ที่ `h = 1` กว้าง 2.618 และกว้างขึ้นเรื่อย ๆ ตาม `h`
- [ ] `temp_monthly` train 235 เดือน test 24 เดือน และ `(M,Ad,M)` ได้ RMSE 1.106
- [ ] รายงานได้ว่า seasonal naive แพ้อยู่เท่าไร ไม่ใช่แค่ชื่อผู้ชนะ
- [ ] ตอบคำถามทุกข้อในทุก markdown cell ที่ขึ้นต้นด้วย "เขียนคำตอบ"
- [ ] restart kernel แล้ว run all ผ่านตลอดโดยไม่มี error
- [ ] เปลี่ยนชื่อไฟล์เป็น `lab08_studentid.ipynb` ก่อนส่ง